# This is a sample Jupyter Notebook

Below is an example of a code cell. 
Put your cursor into the cell and press Shift+Enter to execute it and select the next one, or click 'Run Cell' button.

Press Double Shift to search everywhere for classes, files, tool windows, actions, and settings.

To learn more about Jupyter Notebooks in PyCharm, see [help](https://www.jetbrains.com/help/pycharm/ipython-notebook-support.html).
For an overview of PyCharm, go to Help -> Learn IDE features or refer to [our documentation](https://www.jetbrains.com/help/pycharm/getting-started.html).


---

## 整体场景设定

你们模拟一个 **"SmartNest" 智能家居平台**，包含以下组件：

**中心 Hub**：树莓派（运行 MQTT Broker + Web 管理面板）
**设备节点**：2-3 块 ESP32，分别模拟智能门锁、温度传感器、摄像头网关
**通信协议**：MQTT（设备↔Hub）、HTTP/HTTPS（用户↔Web 面板）、BLE（设备配对）
**云端**：一个简单的 Flask/Node 后端，模拟远程访问和数据存储

---

## Coursework 1：威胁建模 & 攻击模拟

### 威胁 1：MQTT 中间人攻击（MitM on MQTT）

**原理**：很多智能家居设备用的是未加密的 MQTT（端口 1883），攻击者在同一网络可以嗅探和篡改消息。

**具体实现步骤**：
1. 在树莓派上部署 Mosquitto Broker，**故意不开 TLS**（模拟真实中低端产品的常见配置）
2. ESP32 设备发布传感器数据到 `home/temperature`，订阅 `home/lock/command` 等 topic
3. 攻击机（Kali Linux 或普通笔记本）用 `ettercap` 或 `arpspoof` 做 ARP 欺骗，将自己插入 Hub 和设备之间
4. 用 Wireshark 抓包，展示明文 MQTT 消息（payload 里的温度数据、开锁指令一览无余）
5. 用 `mosquitto_pub` 伪造一条开锁指令：`mosquitto_pub -t home/lock/command -m "UNLOCK"` → 门锁 ESP32 收到后执行开锁

**交付物**：Wireshark 截图、攻击脚本、攻击链流程图

### 威胁 2：Web 管理面板漏洞（认证绕过 + 注入）

**原理**：智能家居 Hub 通常有一个 Web 管理界面，如果开发不当，容易存在弱认证、SQL 注入、XSS 等问题。

**具体实现步骤**：
1. 在树莓派上用 Flask 搭一个简易管理面板（设备列表、用户登录、固件更新页面）
2. **故意留漏洞**：默认密码 admin/admin、SQL 注入点（登录表单）、未做 CSRF 保护的设备控制接口
3. 演示攻击：用 `sqlmap` 跑注入拿到用户表、用默认密码登录、通过 CSRF 构造恶意页面远程控制设备
4. 可选加分项：演示 XSS 存储型攻击——在设备名称字段注入脚本，管理员查看时触发

**交付物**：漏洞利用脚本、sqlmap 输出截图、CSRF PoC HTML 页面

### 威胁 3（可选加分）：固件逆向 / BLE 嗅探

如果团队有余力，可以用 `binwalk` 对 ESP32 固件做逆向，提取硬编码的 WiFi 密码或 API Key；或用 `ubertooth`/`nRF Sniffer` 嗅探 BLE 配对过程。

### 风险矩阵

用一个 5×5 的风险矩阵（Likelihood × Impact）对威胁排优先级，MQTT MitM 是"高可能 × 高影响"，Web 漏洞是"中可能 × 高影响"。

---

## Coursework 2：防御方案

### 防御 1：MQTT over TLS + 设备证书认证

1. 给 Mosquitto 配置 TLS（生成 CA 证书、服务端证书、客户端证书）
2. 每个 ESP32 烧录唯一客户端证书，Broker 开启 `require_certificate true`
3. 演示：攻击者再次尝试嗅探 → Wireshark 只看到加密流量；伪造消息 → 被 Broker 拒绝连接
4. 代码展示证书生成脚本和 ESP32 端 TLS 连接代码

### 防御 2：AI 驱动的入侵检测系统（IDS）

1. 在 Hub 上部署一个轻量 Python IDS 脚本
2. 用正常运行时的 MQTT 流量训练一个基线模型（可以用 Isolation Forest 或简单的统计阈值）
3. 检测异常：突然大量 publish 请求、非正常时间段的开锁指令、陌生 Client ID 连接
4. 触发告警（邮件/Telegram 通知 + Web 面板红色警告）
5. 展示对比：无 IDS 时攻击成功 vs 有 IDS 时 30 秒内检测到并阻断

### 防御 3：Web 面板加固

1. 用参数化查询消除 SQL 注入
2. 实现 CSRF Token
3. 加入 MFA（TOTP，用 `pyotp` 库实现）
4. 演示：加固后再跑 sqlmap → 无果；没有 TOTP 码无法登录

### 防御 4：安全 OTA 固件更新

1. 固件镜像签名（用 RSA/Ed25519），设备端验签后才刷入
2. 演示：篡改过的固件被设备拒绝

---

## 4 人分工建议

| 成员 | 负责模块 | 具体任务 |
|------|---------|---------|
| A | MQTT 攻防 | 搭建 MQTT 环境、MitM 攻击演示、TLS 防御实现 |
| B | Web 安全 | 搭建 Flask 面板、留漏洞+攻击演示、加固+MFA |
| C | IDS 系统 | 流量采集、异常检测模型训练、告警系统 |
| D | 架构 + 集成 | 系统架构设计、ESP32 固件开发、OTA 签名、报告整合 |

每个人的工作互相依赖（比如 C 的 IDS 要接 A 的 MQTT 流量），但又可以独立开发测试，最后集成。

---

## 硬件清单（预估成本 ~£30-50）

ESP32 开发板 ×2-3（约 £5-8 each），树莓派（学校可能有借），面包板+LED（模拟门锁状态），一台笔记本当攻击机就够了。如果没有硬件，**全部可以用 Docker 容器模拟**——Mosquitto 容器 + Python 模拟设备脚本，效果一样好。

---

